# 146 — ORPO Alignment

**What you'll learn:**
- How ORPO (Odds Ratio Preference Optimization) differs from RLHF and DPO
- Why ORPO needs no reference model — and why that matters
- The preference dataset format: `{prompt, chosen, rejected}` triples
- What the `beta` hyperparameter controls
- How to measure preference rate (win rate) before and after training

---
**Source:** `examples/146-orpo-alignment/`  
**Model:** `HuggingFaceTB/SmolLM2-135M-Instruct`  
**Part A** is CPU-safe. **Part B** requires a GPU.

In [ ]:
%pip install -q trl transformers peft datasets torch

---
## Part A — CPU-Safe Concept Demonstrations

All cells below run without a GPU and without downloading any model weights.

### A1 — RLHF vs DPO vs ORPO

All three are *alignment* methods — they teach a model to prefer certain responses over others.  
They differ in **complexity**, **compute cost**, and **whether they need a reference model**.

#### Complexity comparison

| Method | Models needed | Steps | Reference model? | Notes |
|--------|--------------|-------|-----------------|-------|
| **RLHF** | SFT model + reward model + policy | 3 stages | Yes (SFT model as anchor) | Classic approach; requires PPO training loop |
| **DPO** | SFT model | 1 stage | Yes (frozen SFT model) | Simplifies RLHF; reference model must be loaded in memory |
| **ORPO** | Base model | 1 stage | **No** | Combines SFT loss + preference loss in one pass |

#### How ORPO works

ORPO trains on `(prompt, chosen, rejected)` triples with a combined loss:

```
L_ORPO = L_SFT + beta × L_OR

where:
  L_SFT = standard cross-entropy on chosen response
  L_OR  = -log(sigmoid(log_odds_chosen - log_odds_rejected))

  log_odds = log(P(response) / (1 - P(response)))
```

The key insight: instead of comparing against a *frozen reference model* (DPO), ORPO computes the **odds ratio within the current model itself** — no second model needed.

#### Why no reference model?

DPO needs a reference model to compute KL divergence (preventing the policy from drifting too far).  
ORPO avoids this by directly penalizing low odds on chosen vs. rejected **relative to each other** — the model is its own implicit reference at every gradient step.

In [ ]:
# A2 — Inspect the preference dataset format (no GPU, no datasets import needed)

CHOSEN = [
    "Here is a clear, step-by-step solution: First, identify the problem. Then, break it into subproblems.",
    "The answer is 42. This is computed by multiplying 6 by 7.",
    "To reverse a list in Python, use `my_list[::-1]` or `list(reversed(my_list))`.",
    "The capital of France is Paris, which has been the country's capital since 987 AD.",
    "Use `git commit -m 'message'` to commit staged changes with a descriptive message.",
]
REJECTED = [
    "I dunno, maybe try something?",
    "The answer could be many things.",
    "You can do it lots of ways I guess.",
    "France has a capital city somewhere in Europe.",
    "Just run some git command or something.",
]
PROMPTS = [
    "How do you approach solving a complex problem?",
    "What is 6 times 7?",
    "How do you reverse a list in Python?",
    "What is the capital of France?",
    "How do you commit changes in git?",
]

rows = [{"prompt": p, "chosen": c, "rejected": r}
        for p, c, r in zip(PROMPTS, CHOSEN, REJECTED)]

print(f"Preference dataset: {len(rows)} rows\n")
print(f"Schema: {{prompt: str, chosen: str, rejected: str}}\n")

for i, row in enumerate(rows[:3]):
    print(f"--- Row {i} ---")
    print(f"  prompt  : {row['prompt']}")
    print(f"  chosen  : {row['chosen'][:70]}")
    print(f"  rejected: {row['rejected']}")
    print()

print("Pattern: chosen = clear, specific, correct answers")
print("         rejected = vague, unhelpful, or wrong responses")

### A3 — The `beta` Hyperparameter

In ORPO, `beta` controls the **strength of the preference signal** relative to standard SFT loss:

```
L_ORPO = L_SFT + beta × L_OR
```

| `beta` value | Effect |
|-------------|--------|
| `0.0` | Pure SFT — learns to generate chosen responses but ignores rejected ones |
| `0.1` (default) | Light alignment — good starting point, preserves fluency |
| `0.5` | Strong alignment — model strongly avoids rejected patterns |
| `1.0+` | Very aggressive — may hurt fluency and coherence |

**Practical guidance:**
- Start at `beta=0.1` (TRL's default)
- If the model still generates bad responses → increase beta
- If fluency/coherence degrades → decrease beta
- Beta is analogous to the KL-penalty coefficient in DPO and PPO

In [ ]:
# A4 — Demonstrate preference rate (win-rate) logic — pure Python, no model
#
# eval_preference_rate() in tools.py measures:
#   "For each (chosen, rejected) pair, does the model assign lower loss to chosen?"
#   win_rate = count(loss_chosen < loss_rejected) / n_samples
#
# Here we simulate it with fake losses to show the concept.

import random
random.seed(42)

def mock_eval_preference_rate(fake_loss_pairs, label=""):
    """
    fake_loss_pairs: list of (loss_chosen, loss_rejected)
    A 'win' is when the model assigns lower loss to the chosen response.
    """
    wins = sum(1 for lc, lr in fake_loss_pairs if lc < lr)
    rate = wins / len(fake_loss_pairs)
    print(f"{label}")
    print(f"  {'Prompt':<6} {'loss_chosen':<14} {'loss_rejected':<14} {'Win?'}")
    for i, (lc, lr) in enumerate(fake_loss_pairs):
        win = "YES" if lc < lr else "no"
        print(f"  {i:<6} {lc:<14.3f} {lr:<14.3f} {win}")
    print(f"  Preference rate (win rate): {wins}/{len(fake_loss_pairs)} = {rate:.1%}\n")
    return rate


# Before training: model is random, roughly 50% win rate
before_pairs = [
    (2.31, 2.18),  # lose — model assigned lower loss to rejected
    (1.95, 2.40),  # win
    (2.55, 2.10),  # lose
    (2.20, 2.35),  # win
    (2.80, 2.70),  # lose
]

# After ORPO training: model learns to prefer chosen responses
after_pairs = [
    (1.20, 2.18),  # win — chosen loss dropped substantially
    (1.10, 2.40),  # win
    (1.35, 2.10),  # win
    (1.05, 2.35),  # win
    (1.45, 2.70),  # win
]

rate_before = mock_eval_preference_rate(before_pairs, "BEFORE ORPO training:")
rate_after  = mock_eval_preference_rate(after_pairs,  "AFTER ORPO training:")

delta = rate_after - rate_before
print(f"Improvement: {rate_before:.1%} → {rate_after:.1%}  ({delta:+.1%})")
print()
print("In tools.py, eval_preference_rate() does the same thing but with REAL model losses.")
print("It runs actual forward passes on (prompt+chosen) and (prompt+rejected) to get loss values.")

---
## Part B — Full ORPO Training Run

> **GPU REQUIRED**  
> ORPO training with a language model requires a CUDA GPU with at least **4 GB VRAM**.  
> Free option: [Google Colab T4 runtime](https://colab.research.google.com) — Runtime > Change runtime type > T4 GPU.  
> Cells will raise an error on CPU-only machines (that is expected).

In [ ]:
# B1 — GPU check
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected.\n"
        "Part B requires a GPU. On Colab: Runtime > Change runtime type > T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

In [ ]:
# B2 — Full ORPO training (from workflow.py)
import sys
sys.path.insert(0, '/content')  # Colab path; adjust if running locally

from src.workflow import create_workflow

workflow = create_workflow()

state = {
    "model_name": "HuggingFaceTB/SmolLM2-135M-Instruct",
    "n_train_pairs": 20,
    "max_steps": 30,
    "preference_rate_before": 0.0,
    "preference_rate_after": 0.0,
    "final_loss": 0.0,
}

print(f"ORPO alignment on {state['model_name']}")
print(f"Training steps: {state['max_steps']}, pairs: {state['n_train_pairs']}\n")
result = workflow(state)

before = result['preference_rate_before']
after  = result['preference_rate_after']
delta  = after - before

print(f"Preference rate before : {before:.1%}")
print(f"Preference rate after  : {after:.1%}")
print(f"Improvement            : {delta:+.1%}")
print(f"Final loss             : {result['final_loss']}")